# JudgeCheck Tutorial 1: Explore MT-Bench & Fit Your First GRM

**Goal:** Load LLM judge ratings, understand the data structure, and fit a **Graded Response Model (GRM)** to learn which MT-Bench questions best separate model quality.

### IRT mapping

| Term | Here |
|---|---|
| Item | MT-Bench question + turn |
| Person | Judge (human) or comparison slot (GPT-4 pairwise) |
| θ | Judge ability (human only) |
| *a* (discrimination) | How sharply the item separates responses |

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from judgecheck.data import load_mt_bench_judgments, summarize_dataset, prepare_grm_matrix, prepare_gpt4_comparison_matrix
from judgecheck.grm import fit_grm, grm_results_to_frame, compare_judges
from judgecheck.viz import plot_item_discrimination, plot_discrimination_comparison

sns.set_theme(style="whitegrid", context="notebook")
%matplotlib inline

## 1. Load the MT-Bench human judgment dataset

Source: [`lmsys/mt_bench_human_judgments`](https://huggingface.co/datasets/lmsys/mt_bench_human_judgments)

Two splits:
- **`human`** — 65 expert annotators, ~3.3K pairwise comparisons
- **`gpt4_pair`** — GPT-4 as automated judge, ~2.4K comparisons

Each row compares **model A vs model B** on one MT-Bench question. This is *not* a direct 1–5 score — we'll map pairwise outcomes to a 3-level ordinal scale for GRM.

In [ ]:
human_df = load_mt_bench_judgments("human")
gpt4_df = load_mt_bench_judgments("gpt4_pair")

pd.concat([
    summarize_dataset(human_df, "human"),
    summarize_dataset(gpt4_df, "gpt4_pair"),
], ignore_index=True)

In [ ]:
# Peek at one judgment row
human_df[["question_id", "turn", "model_a", "model_b", "winner", "judge", "ordinal"]].head(3)

## 2. Build response matrices for GRM

`girth` expects shape **(n_items, n_participants)** with integer ordinal responses.

- **Human judges:** columns = annotators, rows = benchmark items
- **GPT-4:** each item has 15 model-pair comparisons → we use 15 comparison slots as pseudo-participants

In [ ]:
human_matrix, human_items, human_judges = prepare_grm_matrix(human_df)
gpt4_matrix, gpt4_items, gpt4_slots = prepare_gpt4_comparison_matrix(gpt4_df)

print(f"Human: {human_matrix.shape[0]} items × {human_matrix.shape[1]} judges")
print(f"GPT-4: {gpt4_matrix.shape[0]} items × {gpt4_matrix.shape[1]} comparison slots")
print(f"Missing rate (human): {pd.isna(human_matrix).mean():.1%}")

## 3. Fit Graded Response Models

We use **marginal maximum likelihood** via the `girth` package. The GRM estimates, for each benchmark item:

- **Discrimination (a):** slope — how strongly responses change with underlying quality
- **Thresholds (b):** boundaries between ordinal categories

In [ ]:
human_grm = fit_grm(
    human_matrix, human_items, human_judges,
    judge_label="human_experts", estimate_abilities=True,
)
gpt4_grm = fit_grm(gpt4_matrix, gpt4_items, gpt4_slots, judge_label="gpt4_judge")

compare_judges(human_grm, gpt4_grm)

In [ ]:
human_params = grm_results_to_frame(human_grm)
human_params.head(10)

## 4. Visualize item discrimination

These charts answer: **Which MT-Bench questions are the sharpest probes for each judge system?**

In [ ]:
plot_item_discrimination(human_grm, top_n=15)
plt.show()

plot_discrimination_comparison(human_grm, gpt4_grm)
plt.show()

## 5. Takeaways & next steps

1. **Discrimination varies widely across MT-Bench items** — not every question is equally informative for judges.
2. **Compare human vs GPT-4** — if discrimination patterns diverge, the automated judge may be using different cues.
3. **Modeling caveat** — pairwise preferences are mapped to 3 ordinal levels; true 1–5 GRM fits need absolute score data.

**Next session ideas:** estimate judge ability (θ), test information curves, and add more judge models (Claude, Llama, etc.).

## 6. Labeled results, judge ability & HTML report

The full pipeline (`python scripts/fit_grm.py`) now also:

- Joins **real question text** from MT-Bench onto every item
- Estimates **human judge ability (θ)** — who is most consistent?
- Summarizes discrimination by **category** (Writing, Math, …)
- Writes **`outputs/report.html`** and **`outputs/SUMMARY.txt`**

See **[docs/GUIDE.md](../docs/GUIDE.md)** (results) and **[AGENTS.md](../AGENTS.md)** (development).

In [ ]:
from judgecheck.data import build_item_catalog, enrich_with_labels
from judgecheck.grm import judge_abilities_to_frame, summarize_by_category

catalog = build_item_catalog()
labeled = enrich_with_labels(human_params, catalog)
abilities = judge_abilities_to_frame(human_grm)
categories = summarize_by_category(labeled)

print("Top 3 sharpest questions:")
display(labeled[["short_label", "discrimination"]].head(3))
print("\nTop 5 most consistent human judges:")
display(abilities.head())
print("\nDiscrimination by category:")
display(categories)